# 03. Threat Detection Demo
Simulate real-time threat detection alert generation.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.threat_detector import ThreatDetector
from src.alert_generator import generate_alerts, print_alert_summary
from src.preprocessor import preprocess
from src.feature_engineering import select_top_features, get_feature_importances
from src.model_trainer import train_random_forest, train_isolation_forest

In [ ]:
df = pd.read_csv('../data/raw/synthetic_network_traffic.csv')
data = preprocess(df)
imp = get_feature_importances(data['X_train'], data['y_train'], data['feature_names'])
X_tr_sel, X_te_sel, feats = select_top_features(data['X_train'], data['X_test'], data['feature_names'], imp, 20)

rf_model = train_random_forest(X_tr_sel, data['y_train'], n_estimators=20)
if_model = train_isolation_forest(X_tr_sel, contamination=0.20)

detector = ThreatDetector()
detector.set_models(rf_model, if_model, data['scaler'], feature_names=feats)

## Real-Time Predictions

In [ ]:
sample_normal = X_te_sel[data['y_test'] == 0][0].reshape(1, -1)
res_normal = detector.predict(sample_normal, mode='combined', pre_scaled=True)
print('Normal Sample Result:', res_normal['threat_label'][0])

sample_attack = X_te_sel[data['y_test'] == 1][0].reshape(1, -1)
res_attack = detector.predict(sample_attack, mode='combined', pre_scaled=True)
print('Attack Sample Result:', res_attack['threat_label'][0])